# Schematic2SDN - continue training the circuit vision adapter (Colab)

Continues the supplied Qwen2.5-VL-7B LoRA adapter on audited circuit image/netlist pairs, then measures
the **old and new adapter on the same held-out test split** so you can see whether training helped.

**Before you start**
1. Runtime -> Change runtime type -> **GPU** (T4 works; L4/A100 is faster).
2. Upload `Schematic2SDN_colab.zip` to the root of your Google Drive (*My Drive*).

Checkpoints are written to Drive (`My Drive/S2SDN_runs/`), so a disconnect does not lose finished epochs.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'
BF16 = torch.cuda.is_bf16_supported()
FREE_GIB = torch.cuda.mem_get_info()[0] / 2**30
# 7b = BF16 GPUs (L4/A100); 7b-t4 = FP16 on a 16 GB T4. Both continue the supplied 7B adapter.
PROFILE = '7b' if BF16 and FREE_GIB >= 12 else '7b-t4'
print(torch.cuda.get_device_name(), f'BF16={BF16} free={FREE_GIB:.1f} GiB -> profile {PROFILE}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, time, zipfile
BUNDLE = '/content/drive/MyDrive/Schematic2SDN_colab.zip'
assert os.path.exists(BUNDLE), 'Upload Schematic2SDN_colab.zip to My Drive first.'
zipfile.ZipFile(BUNDLE).extractall('/content')
ROOT = '/content/Schematic2SDN'
%cd {ROOT}
RUN = f"/content/drive/MyDrive/S2SDN_runs/run-{time.strftime('%Y%m%d-%H%M%S')}"
os.makedirs(RUN, exist_ok=True)
DATASET = f'{ROOT}/training/prepared_amsnet'   # switch to training/prepared_samples when present
BASE_ADAPTER = f'{ROOT}/checkpoints/circuit-lora/adapter'
print('run folder:', RUN)
!wc -l {DATASET}/*.jsonl

In [ ]:
# Training stack + the backend modules the evaluator imports (unpinned to fit Colab's torch).
!pip -q install -U "transformers>=4.57,<6" "peft>=0.17,<1" "accelerate>=1.10,<2" "bitsandbytes>=0.47,<1"
!pip -q install fastapi uvicorn httpx pydantic python-dotenv itsdangerous pymupdf python-docx opencv-python-headless scipy

## 1. Baseline: the supplied adapter on the held-out test split
Loads the model once (`S2SDN_KEEP_LOADED=1`) and scores up to `LIMIT` test images.

In [ ]:
import subprocess, time, urllib.request
LIMIT = 20
def serve(adapter):
    env = {**os.environ, 'S2SDN_ADAPTER': adapter, 'S2SDN_MODEL_TAG': 'schematic2sdn-qwen-vl', 'S2SDN_PROFILE': PROFILE, 'S2SDN_KEEP_LOADED': '1'}
    p = subprocess.Popen(['python', '-m', 'uvicorn', 'serve:app', '--app-dir', 'training', '--host', '127.0.0.1', '--port', '11435'], env=env)
    for _ in range(60):
        try: urllib.request.urlopen('http://127.0.0.1:11435/api/tags'); return p
        except Exception: time.sleep(2)
    p.kill(); raise RuntimeError('adapter server did not start')
def evaluate(adapter, out):
    p = serve(adapter)
    try:
        env = {**os.environ, 'IMAGE_MODE': 'vision', 'VISION_MODEL': 'schematic2sdn-qwen-vl', 'VISION_OLLAMA_URL': 'http://127.0.0.1:11435', 'VISION_TIMEOUT_SECONDS': '900'}
        subprocess.run(['python', 'training/evaluate.py', '--dataset', DATASET, '--output', out, '--limit', str(LIMIT)], env=env, check=True)
    finally:
        p.kill(); p.wait()
    import json; r = json.load(open(out))
    print(out, '-> exact graph matches', r['exact_graph_matches'], '/', r['samples'], 'failed', r['failed'])
    return r
baseline = evaluate(BASE_ADAPTER, f'{RUN}/eval-baseline.json')

## 2. Smoke test (one step on the longest sample: checks GPU memory headroom)

In [ ]:
!python training/finetune.py --profile {PROFILE} --dataset {DATASET} --output {RUN}/train --init-adapter {BASE_ADAPTER} --smoke-test

## 3. Continue training from the supplied adapter

In [ ]:
EPOCHS = 2
!python training/finetune.py --profile {PROFILE} --dataset {DATASET} --output {RUN}/train --init-adapter {BASE_ADAPTER} --epochs {EPOCHS}
!cat {RUN}/train/training-report.json

## 4. Same test split, new adapter

In [ ]:
trained = evaluate(f'{RUN}/train/adapter', f'{RUN}/eval-trained.json')
print(f"baseline {baseline['exact_graph_matches']}/{baseline['samples']}  ->  trained {trained['exact_graph_matches']}/{trained['samples']}")

## 5. Bring the adapter home
`My Drive/S2SDN_runs/<run>/train/adapter` holds the new adapter. Download it and replace
`checkpoints/circuit-lora/adapter` on your PC **only if** the trained score beats the baseline.

In [ ]:
!cd {RUN}/train && zip -qr {RUN}/adapter.zip adapter
print('Download:', f'{RUN}/adapter.zip')